In [1]:
from vllm import LLM, SamplingParams
import base64
import os
import fitz  # PyMuPDF
import mimetypes
import gc
import torch
from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment

# Global LLM instance
llm = None

def init_llm():
    global llm
    if llm is None:
        print("Initializing VLLM...")
        llm = LLM(
            # model="Qwen/Qwen3.5-4B",
            # model="Qwen/Qwen3.5-27B-FP8",
            model="Qwen/Qwen3.5-9B",
            tensor_parallel_size=1,
            max_model_len=262144,
            gpu_memory_utilization=0.8,
            trust_remote_code=True,
            # cpu_offload_gb=10
            # reasoning_parser="qwen3"
        )

def encode_image(image_path):
    """Encodes a file from disk to base64."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def prepare_page_messages(base64_image, mime_type="image/png", extracted_images=None):
    """Prepares the message list for a single page."""
    
    image_section = ""
    if extracted_images:
        img_list = "\n".join([f"- {os.path.basename(img)}" for img in extracted_images])
        image_section = f"""
**Images:**
The following images have been extracted from this page. If they are figures, charts, or diagrams relevant to the content, insert them at the appropriate location using `![Description](extracted_images/<filename>)`.
Do NOT include logos, icons, or decorative elements.
Available images:
{img_list}
"""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{mime_type};base64,{base64_image}"
                    }
                },
                {
                    "type": "text",
                    "text": f"""Transcribe the main article body from this image into Markdown.

**Include:**
- The main text content
- **Tables**: Convert all tables into standard Markdown tables.
{image_section}

**Exclude (Do NOT transcribe):**
- The header at the top
- The entire metadata (e.g., "Target Article", citation details, dates, "Keywords", and the "What is Open Peer Commentary?" box)
- The footer (e.g., "© Cambridge University Press 2019", logos)

Output ONLY the transcribed text. Do not include explanations or any other text."""
                }
            ]
        }
    ]
    return messages

def smart_join_pages(pages):
    if not pages:
        return ""
    
    merged_text = pages[0].strip()
    
    for page in pages[1:]:
        page = page.strip()
        if not page: continue
        
        # Check if the previous page ended with a hyphen (split word)
        if merged_text.endswith("-"):
            # Remove hyphen and join without space
            merged_text = merged_text[:-1] + page
        # Check if the previous page ended with sentence-ending punctuation
        elif merged_text[-1] not in ".!?\":":
            # Likely a mid-sentence split; join with a space
            merged_text += " " + page
        else:
            # Standard paragraph break
            merged_text += "\n\n" + page
            
    return merged_text

def extract_images_from_page(doc, page, page_index, output_dir="extracted_images"):
    """Extracts images from a PDF page and saves them."""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    image_list = page.get_images(full=True)
    saved_images = []
    
    for img_index, img in enumerate(image_list):
        xref = img[0]
        try:
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]
            
            image_filename = f"image_page{page_index + 1}_{img_index + 1}.{image_ext}"
            image_path = os.path.join(output_dir, image_filename)
            
            with open(image_path, "wb") as f:
                f.write(image_bytes)
            
            # Return the path that the LLM should use
            saved_images.append(image_path)
        except Exception as e:
            print(f"Failed to extract image {img_index} on page {page_index}: {e}")
            
    return saved_images


/home/dw/github/wake-manuscript/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Init LLM early
try:
    init_llm()
except Exception as e:
    print(f"Failed to initialize VLLM: {e}")

Initializing VLLM...
INFO 03-05 12:06:02 [utils.py:238] non-default args: {'trust_remote_code': True, 'max_model_len': 262144, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'Qwen/Qwen3.5-9B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-05 12:06:03 [model.py:530] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 03-05 12:06:03 [model.py:1553] Using max model len 262144


2026-03-05 12:06:03,698	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 03-05 12:06:03 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-05 12:06:04 [config.py:232] Setting attention block size to 528 tokens to ensure that attention page size is >= mamba page size.
INFO 03-05 12:06:04 [config.py:263] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
INFO 03-05 12:06:04 [vllm.py:747] Asynchronous scheduling is enabled.


...

WARNING 03-05 12:06:14 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=47558) INFO 03-05 12:06:15 [core.py:101] Initializing a V1 LLM engine (v0.16.1rc1.dev281+g612e7729c) with config: model='Qwen/Qwen3.5-9B', speculative_config=None, tokenizer='Qwen/Qwen3.5-9B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=262144, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=Struc

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:01,  2.83it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:00<00:00,  2.75it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:01<00:00,  2.71it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  3.00it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:01<00:00,  2.90it/s]
(EngineCore_DP0 pid=47558) 


(EngineCore_DP0 pid=47558) INFO 03-05 12:08:35 [default_loader.py:293] Loading weights took 1.41 seconds
(EngineCore_DP0 pid=47558) INFO 03-05 12:08:35 [gpu_model_runner.py:4344] Model loading took 17.66 GiB memory and 132.495748 seconds
(EngineCore_DP0 pid=47558) INFO 03-05 12:08:35 [gpu_model_runner.py:5260] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore_DP0 pid=47558) INFO 03-05 12:08:41 [backends.py:913] Using cache directory: /home/dw/.cache/vllm/torch_compile_cache/159c6efef5/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=47558) INFO 03-05 12:08:41 [backends.py:973] Dynamo bytecode transform time: 2.37 s
(EngineCore_DP0 pid=47558) INFO 03-05 12:08:42 [backends.py:370] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=47558) INFO 03-05 12:08:46 [backends.py:386] Compiling a graph for compile range (1, 8192) takes 4.72 s
(EngineCore_DP0 pid=47558) IN

(EngineCore_DP0 pid=47558) 2026-03-05 12:08:48,551 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=47558) 2026-03-05 12:08:48,610 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 40/40 [00:01<00:00, 34.14it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 30.39it/s]


(EngineCore_DP0 pid=47558) INFO 03-05 12:08:51 [gpu_model_runner.py:5366] Graph capturing finished in 3 secs, took -0.88 GiB
(EngineCore_DP0 pid=47558) INFO 03-05 12:08:51 [core.py:282] init engine (profile, create kv cache, warmup model) took 16.11 seconds
(EngineCore_DP0 pid=47558) INFO 03-05 12:08:51 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 03-05 12:08:51 [llm.py:388] Supported tasks: ['generate']
(EngineCore_DP0 pid=47558) INFO 03-05 12:08:51 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 03-05 12:08:51 [llm.py:388] Supported tasks: ['generate']


In [3]:
input_path = 'rational.pdf'
max_pages = 3

# Guess the MIME type based on extension
mime_type, _ = mimetypes.guess_type(input_path)

messages_batch = []

if mime_type == 'application/pdf':
    print(f"Processing PDF: {input_path}")
    try:
        doc = fitz.open(input_path)
        total_pages = max_pages if max_pages else len(doc)
        
        print(f"Preparing {total_pages} pages for transcription...")
        for i, page in enumerate(doc[:total_pages]):
            
            # Extract images from the page first
            page_images = extract_images_from_page(doc, page, i)
            
            # Render page to image (zoom=3 for high resolution ~216 DPI)
            zoom = 3.0
            mat = fitz.Matrix(zoom, zoom)
            pix = page.get_pixmap(matrix=mat)
            img_bytes = pix.tobytes("png")
            
            # Encode bytes directly without saving to disk
            base64_img = base64.b64encode(img_bytes).decode('utf-8')
            
            # Prepare message for this page
            messages = prepare_page_messages(base64_img, "image/png", extracted_images=page_images)
            messages_batch.append(messages)
            
        doc.close()
    except Exception as e:
        print(f"Error processing PDF: {e}")
        import traceback
        traceback.print_exc()

elif mime_type and mime_type.startswith('image'):
    print(f"Processing Image: {input_path}")
    try:
        base64_img = encode_image(input_path)
        messages = prepare_page_messages(base64_img, mime_type)
        messages_batch.append(messages)
    except Exception as e:
        print(f"Error processing image: {e}")
    
else:
    print(f"Unsupported or unrecognized file type: {mime_type} for {input_path}")

# Run Batch Inference
print(f"Running VLLM batch inference on {len(messages_batch)} pages...")

# Using parameters similar to original script
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.8,
    # presence_penalty=1.5,
    top_k=20,
    max_tokens=32768
)
# print('messages_batch', messages_batch)
outputs = []
for message in messages_batch:
    out = llm.chat(
        messages=message, 
        sampling_params=sampling_params, 
        # chat_template_kwargs={"enable_thinking": False}
    )
    outputs.append(out[0])

transcribed_texts = []
for output in outputs:
    generated_text = output.outputs[0].text
    
    # Remove text before </think> if present
    if "</think>" in generated_text:
        generated_text = generated_text.split("</think>")[-1].strip()
        
    print('*'*50)
    print(generated_text)
    print('*'*50)
    transcribed_texts.append(generated_text)

# Combine all pages using smart stitching
final_markdown = smart_join_pages(transcribed_texts)

Processing PDF: rational.pdf
Preparing 3 pages for transcription...
Running VLLM batch inference on 3 pages...


Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 03-05 12:09:16 [hf.py:318] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 1/1 [01:43<00:00, 103.97s/it, est. speed input: 41.83 toks/s, output: 85.21 toks/s]

**************************************************
# Resource-rational analysis: Understanding human cognition as the optimal use of limited computational resources

Falk Lieder$^{a}$ and Thomas L. Griffiths$^{b}$

$^{a}$Max Planck Institute for Intelligent Systems, Tübingen 72076, Germany and $^{b}$Departments of Psychology and Computer Science, Princeton University, Princeton, New Jersey 08544, USA
falk.lieder@tuebingen.mpg.de https://re.is.mpg.de
tomg@princeton.edu https://psych.princeton.edu/person/tom-griffiths

## Abstract

Modeling human cognition is challenging because there are infinitely many mechanisms that can generate any given observation. Some researchers address this by constraining the hypothesis space through assumptions about what the human mind can and cannot do, while others constrain it through principles of rationality and adaptation. Recent work in economics, psychology, neuroscience, and linguistics has begun to integrate both approaches by augmenting rational 

In [7]:
import textwrap

print(textwrap.fill(out[0].outputs[0].text, width=100))


The user wants me to transcribe the main article body from the provided image into Markdown.  **1.
Analyze the Image:** *   **Header:** "Lieder and Griffiths: Resource-rational analysis" (This is
likely a running head, I should exclude it based on instructions). *   **Page Number:** "3" (Top
right, exclude). *   **Main Text:**     *   Starts with "that decision-makers select the action
$a^\star$..."     *   Contains Equation (1).     *   Discusses utility functions, probability, and
the history of rationality (Von Neumann & Morgenstern, Edwards, Newell, etc.).     *   Mentions
cognitive biases (Tversky & Kahneman, Wason, etc.).     *   Discusses "bounded rationality" (Simon,
Tversky & Kahneman).     *   Discusses "resource-rational analysis" (Chater & Oaksford).     *
References Figure 1.     *   References Box 1 (though Box 1 text is cut off at the bottom, I will
transcribe what is visible). *   **Figure 1:** A diagram showing a triangle representing
"Optimality", "Resource rationalit

In [5]:
'think' in out[0].outputs[0].text

True

In [11]:
print(outputs[2].outputs[0].text.split("</think>")[-1].strip())

that decision-makers select the action $a^\star$ that maximizes their expected utility (Von Neumann & Morgenstern 1944), that is

$a^\star = \arg \max_a \int u(o) \cdot p(o|a) \, do,$

where the utility function $u$ measures how good the outcome $o$ is from the decision-maker's perspective and $p(o|a)$ is the conditional probability of its occurrence if action $a$ is taken.

![Resource rationality and its relationship to optimality and Tversky and Kahneman's concept of bounded rationality. The horizontal dimension corresponds to alternative cognitive mechanisms that achieve the same level of performance. Each dot represents a possible mind. The gray dots are minds with bounded cognitive resources and the blue dots are minds with unlimited computational resources. The thick black line symbolizes the bounds entailed by people's limited cognitive resources. Resource limitations reflect anatomical, physiological, and metabolic constraints on neural information processing as discussed below

In [31]:
final_markdown

'# Resource-rational analysis: Understanding human cognition as the optimal use of limited computational resources\n\n**Falk Lieder**<sup>a</sup> and **Thomas L. Griffiths**<sup>b</sup>\n\n<sup>a</sup>Max Planck Institute for Intelligent Systems, Tübingen 72076, Germany and <sup>b</sup>Departments of Psychology and Computer Science, Princeton University, Princeton, New Jersey 08544, USA\n\nfalk.lieder@tuebingen.mpg.de https://re.is.mpg.de\ntomg@princeton.edu https://psych.princeton.edu/person/tom-griffiths\n\n## Abstract\n\nModeling human cognition is challenging because there are infinitely many mechanisms that can generate any given observation. Some researchers address this by constraining the hypothesis space through assumptions about what the human mind can and cannot do, while others constrain it through principles of rationality and adaptation. Recent work in economics, psychology, neuroscience, and linguistics has begun to integrate both approaches by augmenting rational models

In [12]:
with open('output.md', "w") as f:
    f.write(final_markdown)